In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd

In [3]:
!pip install wandb -q

In [4]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
import joblib

In [6]:
wandb.init(
    project="23f1000054-t22026",
    name="tfidf_baseline"
)

In [7]:
train = pd.read_csv(
    "/content/drive/MyDrive/SmartMCQ/train.csv"
)

test = pd.read_csv(
    "/content/drive/MyDrive/SmartMCQ/test.csv"
)

print(train.shape)
print(test.shape)

(2000, 8)
(500, 7)


In [8]:
rows = []

for _, row in train.iterrows():

    prompt = row["prompt"]
    correct_answer = row["answer"]

    for option in ["A", "B", "C", "D", "E"]:

        rows.append({
            "text": prompt + " [SEP] " + str(row[option]),
            "label": 1 if option == correct_answer else 0
        })

binary_train = pd.DataFrame(rows)

binary_train.head()

,text,label
0,Pick the best possible answer: What is Martin ...,0
1,Pick the best possible answer: What is Martin ...,1
2,Pick the best possible answer: What is Martin ...,0
3,Pick the best possible answer: What is Martin ...,0
4,Pick the best possible answer: What is Martin ...,0


In [9]:
print(binary_train.shape)

binary_train["label"].value_counts()

(10000, 2)


,count
label,
0,8000
1,2000


In [10]:
X_train, X_valid, y_train, y_valid = train_test_split(
    binary_train["text"],
    binary_train["label"],
    test_size=0.2,
    random_state=42,
    stratify=binary_train["label"]
)

print(X_train.shape)
print(X_valid.shape)

(8000,)
(2000,)


In [11]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    stop_words="english"
)

X_train_vec = tfidf.fit_transform(X_train)
X_valid_vec = tfidf.transform(X_valid)

print(X_train_vec.shape)
print(X_valid_vec.shape)

(8000, 11343)
(2000, 11343)


In [12]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train_vec, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, n_jobs=-1)

In [13]:
preds = model.predict(X_valid_vec)

In [14]:
accuracy = accuracy_score(y_valid, preds)
precision = precision_score(y_valid, preds)
recall = recall_score(y_valid, preds)
f1 = f1_score(y_valid, preds)

In [15]:
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

Accuracy : 0.9195
Precision: 0.7394789579158316
Recall   : 0.9225
F1 Score : 0.8209121245828699


In [16]:
print(
    classification_report(
        y_valid,
        preds
    )
)

              precision    recall  f1-score   support

           0       0.98      0.92      0.95      1600
           1       0.74      0.92      0.82       400

    accuracy                           0.92      2000
   macro avg       0.86      0.92      0.88      2000
weighted avg       0.93      0.92      0.92      2000



In [17]:
wandb.log({
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1
})

In [18]:
joblib.dump(
    model,
    "tfidf_model.pkl"
)

joblib.dump(
    tfidf,
    "tfidf_vectorizer.pkl"
)

['tfidf_vectorizer.pkl']

In [19]:
wandb.finish()

accuracy,▁
f1,▁
precision,▁
recall,▁
accuracy,0.9195
f1,0.82091
precision,0.73948
recall,0.9225


In [20]:
joblib.dump(
    model,
    "/content/drive/MyDrive/SmartMCQ/tfidf_model.pkl"
)

joblib.dump(
    tfidf,
    "/content/drive/MyDrive/SmartMCQ/tfidf_vectorizer.pkl"
)

print("Saved Successfully")

Saved Successfully
